# **EfficientNet Pre-trained Binary Model with Transformed Input**

In this notebook we test a pre-trained EfficientNet model to predict Animal vs Non-animal. <br>
For this, transformed/pre-processed images were used, which were generated with the help of the notebook `cleaner_routine.ipynb`. <br>

If you have ran any other model before, do not forget to reset the `input_image_directory` by running 'reset input' in the terminal.

## Libraries

In [ ]:
import os
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

In [ ]:
# Libraries
import pandas as pd

from deep.constants import ROOT, METADATA_FILE, SEEDS, SCORES
from deep.modelling.model_runs import run_binary_model
from deep.modelling.model_specifications import efficient_net

In [ ]:
# Setting options
pd.set_option('display.max_rows', None)

## Data Loading

In [ ]:
# Load the metadata
data = pd.read_csv(METADATA_FILE)

# Drop unnecessary columns for this problem
data.drop(columns=['phylum', 'family'], inplace=True)

## Data Preprocessing

In [ ]:
# Define seeds and path
binary_baseline_path = f"{SCORES}/binary_scores.json"

### Clean Images

In [ ]:
# Create transformed images
notebook_path = f'"{ROOT}/notebooks/preprocess_routine/cleaner_routine.ipynb"'
%run $notebook_path

### Resize Images

In [ ]:
notebook_path = f'"{ROOT}/notebooks/preprocess_routine/resize_routine.ipynb"'
%run $notebook_path

## Data Modelling

### Raw Model

In [ ]:
run_binary_model(
    file_path=binary_baseline_path
    ,data=data
    ,seed=SEEDS[0]
    ,model=efficient_net(
        regularizer=False
        ,dropout=False
    )
    ,loss='crossentropy'
    ,epochs=10
    ,type='transformed'
    ,model_name='efficientnetb4'
)

This model with the transformed images proved to yield equal precision results than with the original images.

### Model with Regularization

In this section we add L2 regularization to our model and test if it improves the model

In [ ]:
run_binary_model(
    file_path=binary_baseline_path
    ,data=data
    ,seed=SEEDS[0]
    ,model=efficient_net(
        regularizer=True
        ,dropout=False
    )
    ,loss='crossentropy'
    ,epochs=10
    ,type='transformed'
    ,model_name='efficientnetb4'
)

Regularization did not improve the model.

### Model with Dropout

In this section we add a dropout layer to the model and test if it improves the results.

In [ ]:
run_binary_model(
    file_path=binary_baseline_path
    ,data=data
    ,seed=SEEDS[0]
    ,model=efficient_net(
        regularizer=False
        ,dropout=True
    )
    ,loss='crossentropy'
    ,epochs=10
    ,type='transformed'
    ,model_name='efficientnetb4'
)

The dropout layer did not improve the model.

Now, we will attempt to fix the class imbalance based on the transformed images and recheck the model.